In [ ]:
import pandas as pd

## pd.cut(x, bins, labels) — 按数值边界分箱

把连续数据按**给定的区间边界**切成离散段。返回 `Categorical` 类型，自带省内存和分类属性。

### 参数

| 参数 | 说明 |
|---|---|
| `x` | 被切的一维数据：Series、列表、ndarray |
| `bins` | 切割点：整数=等宽分成 N 段；列表=`[0,60,80,100]` 手动定边界；也可以传 `IntervalIndex` |
| `labels` | 每个区间的显示名称，长度必须和区间数一致。不传则显示 `(60, 80]` 这样的区间表示 |
| `right` | 默认 `True`：区间右闭左开 `(a, b]`。改为 `False` 则左闭右开 `[a, b)` |
| `include_lowest` | `right=True` 时，是否把最小值兜进第一个区间（避免被排挤成 NaN） |
| `precision` | 默认 3，控制区间端点显示的小数位数 |
| `retbins` | `True` 时返回 `(结果, 切点数组)`，调试用 |

### 核心记忆

```python
pd.cut(df['成绩'], bins=3)                              # 等宽 3 段
pd.cut(df['成绩'], bins=[0, 60, 80, 100])                # 手动切点
pd.cut(df['成绩'], bins=[0,60,80,100], labels=['C','B','A'])  # 贴标签
```

> **cut 按值域切** → 每个箱的数值跨度相等，但每箱人数不等。
> **qcut 按数量切** → 每箱人数尽量相等，但每箱数值跨度不等（见后文）。


In [ ]:
# ===== pd.cut 按数值边界分箱 =====
import pandas as pd
df = pd.read_csv(r'D:/dsml/data/students.csv')
df1 = df.head(10)
print(df1[['学号', '姓名', 'GPA']])

# bins 为整数 → 等宽分成 n 段
# pandas 自动计算边界，使每段数值跨度相同
result = pd.cut(df1['GPA'], bins=2)
print('\n--- bins=2（等宽 2 段）---')
print(result)

In [ ]:
# 看每段有多少人
pd.cut(df1['GPA'], bins=2).value_counts()

In [ ]:
# bins 为列表 → 手动指定边界
print('自定义分段 [1,2,3,4]:')
print(pd.cut(df1['GPA'], bins=[1, 2, 3, 4], labels=['C', 'B', 'A']))

# 不传 labels 时默认显示区间范围
print('\n不传 labels（显示区间本身）:')
print(pd.cut(df1['GPA'], bins=[1, 2, 3, 4]))

print('\n各区段分布:')
print(pd.cut(df1['GPA'], bins=[1, 2, 3, 4], labels=['C', 'B', 'A']).value_counts())

# 分箱后是 Categorical 类型，可以直接拿编码
cut_result = pd.cut(df1['GPA'], bins=[1, 2, 3, 4], labels=['C', 'B', 'A'])
print('\n类别编码 (.cat.codes):', cut_result.cat.codes.tolist())
print('合法类别 (.cat.categories):', cut_result.cat.categories.tolist())

## pd.qcut(x, q) — 按样本数量分箱

按**分位数**切分，保证每个箱里的样本数尽可能相等。

### 参数

| 参数 | 说明 |
|---|---|
| `x` | 被切的一维数据 |
| `q` | 整数=切成 N 等份（如 `q=3` 切成 3 个箱，每箱约 1/3 的数据）；列表=`[0, 0.25, 0.5, 0.75, 1]` 按分位点切 |
| `labels` | 同 cut，给每个箱起名字 |
| `retbins` | 同 cut，返回切点 |
| `precision` | 同 cut，小数位数 |
| `duplicates` | 遇到重复分位点时怎么办：`'raise'` 报错 / `'drop'` 自动去重 |

### cut vs qcut

| | cut | qcut |
|---|---|---|
| 切分依据 | 数值大小等分 | 样本数量等分 |
| 结果 | 每箱值域等宽，人数随意 | 每箱人数均匀，值域随意 |
| 适用 | 你有明确的业务阈值（如 60 分及格） | 你没有先验阈值，想自然分层 |
| 底层 | 纯区间切分 | 基于分位数 |

```python
pd.qcut(df['成绩'], q=3)                        # 每箱约 1/3 数据
pd.qcut(df['成绩'], q=[0, 0.5, 0.8, 1])         # 按 0~50%, 50%~80%, 80%~100% 三段
```

> **分箱后数据类型**：cut/qcut 返回 `Categorical`，自动享有省内存、`.cat.codes` 拿编码、分组统计更快等好处，无需再 `astype('category')`。


In [ ]:
# ===== pd.qcut 按样本数量分箱 =====
# q=3 → 每箱约 1/3 的数据（人数尽量均匀）
result = pd.qcut(df1['GPA'], q=3)
print(result)
print()
print('各区段人数（基本均匀）:')
print(pd.qcut(df1['GPA'], 3).value_counts())

# 对比 cut: 等宽 vs 等量
print('\n--- 对比 cut(等宽) vs qcut(等量) ---')
print('cut  bins=3  等宽:', pd.cut(df1['GPA'], bins=3).value_counts().tolist())
print('qcut q=3    等量:', pd.qcut(df1['GPA'], q=3).value_counts().tolist())
print('（qcut 每箱人数接近，cut 可能差异很大）')

## df.rename()、df.set_index()、df.reset_index()

### df.rename()
重命名列名或行索引标签。传入 `columns` 字典修改列名，传入 `index` 字典修改索引标签。默认不改变原 df，需用 `inplace=True` 或赋值。

```python
df.rename(columns={"旧列名": "新列名"})
df.rename(index={0: "第一行"})
df.rename(columns=str.lower)  # 也可以用函数批量处理
```

### df.set_index()
将现有的某一列或多列提升为行索引。原索引被替换，默认不改变原 df。常用于把 ID 列、时间列设为索引。

```python
df.set_index("学号")
df.set_index(["城市", "年份"])  # 多列 → 多级索引
```

### df.reset_index()
将当前行索引"降级"回普通列，恢复默认的 0,1,2... 整数索引。多级索引也能一次性全部重置。常用于索引干扰了操作时（如 merge、groupby 后）。

```python
df.reset_index()              # 全部重置
df.reset_index(drop=True)     # 不保留旧索引列，直接丢弃
```


In [ ]:
# ===== df.rename() =====
import pandas as pd
df = pd.read_csv(r'D:/dsml/data/students.csv').head(8)
print('原始列名:', df.columns.tolist())

# 单个改名：列名 学号 -> StudentID
df2 = df.rename(columns={'学号': 'StudentID'})
print('\n改名后:', df2.columns.tolist())

# 批量改名：用字典映射多列
df3 = df.rename(columns={'姓名': 'Name', '专业': 'Major'})
print(df3.columns.tolist())

# 用函数批量处理：列名全部大写
df4 = df.rename(columns=str.upper)
print(df4.columns.tolist())

# 修改行索引标签：第 0 行改名为 'first'
df5 = df.rename(index={0: 'first'})
print(df5.head(3))

In [ ]:
# ===== df.set_index() =====
df = pd.read_csv(r'D:/dsml/data/students.csv').head(8)
print('原始表（默认索引）:')
print(df[['学号','姓名','GPA']])

# 将学号设为索引
df_idx = df.set_index('学号')
print('\n以学号为索引:')
print(df_idx[['姓名','GPA']])

# 多列联合索引
#不等于连续使用set_index set_index是替换操作，会直接剔除当前索引，除非写入append=True变为多级索引
df_multi = df.set_index(['专业', '年级'])
print('\n多级索引（专业+年级）:')
print(df_multi[['姓名','GPA']].head(6))
df_multi = df.set_index('专业').set_index('年级',append=True)
print('\n多级索引（专业+年级）:')
print(df_multi[['姓名','GPA']].head(6))
#df_multi[]只能查最外层
df_multi = df_multi.sort_index()
print(df_multi.index.is_monotonic_increasing)
print(df_multi.loc['化学'])
print(df_multi.loc[('化学','大一')])

In [ ]:
# ===== df.reset_index() =====
df = pd.read_csv(r'D:/dsml/data/students.csv').head(8)
df_idx = df.set_index('学号')
print('有学号索引的表:')
print(df_idx[['姓名','GPA']])

# 重置索引：学号降级为普通列
df_reset = df_idx.reset_index()
print('\nreset_index 后（学号回来了）:')
print(df_reset[['学号','姓名','GPA']])

# drop=True：直接丢弃索引列，不保留
df_drop = df_idx.reset_index(drop=True)
print('\ndrop=True（学号被丢弃，回到默认索引）:')
print(df_drop[['姓名','GPA']])

In [ ]:
# ===== 综合：rename + set_index + reset_index 联动 =====
df = pd.read_csv(r'D:/dsml/data/weather.csv').head(12)
print('原始表，列名带中文和单位:')
print(df.columns.tolist())

# Step 1: rename 把带单位的列名变简洁
df = df.rename(columns={
    '温度(℃)': 'temp',
    '湿度(%)': 'humidity',
    '风速(m/s)': 'wind',
    '降水量(mm)': 'rain'
})
print('\nrename 后:', df.columns.tolist())

# Step 2: set_index 用日期+城市做联合索引
df = df.set_index(['城市', '日期'])
print('\n多级索引（城市+日期）:')
print(df.head(6))

# Step 3: reset_index 还回来
df = df.reset_index()
print('\nreset_index 恢复成平表:')
print(df.head(4))